# Training: U-Net for Binary Burn Detection

**Purpose**: Train the U-Net model on Sentinel-2 imagery for binary burn scar segmentation.

**Prerequisites**:
- Run `01_data_pipeline.ipynb` first to prepare data and loaders

**Workflow**:
1. Define U-Net architecture
2. Set up loss function (BCE + Dice for class imbalance)
3. Train with validation monitoring
4. Save best model checkpoint
5. Plot training curves

**Outputs**:
- Trained model weights (`.pth`)
- Training history (loss, IoU curves)
- Best validation IoU score

**Issues**: #10 Model Architecture (D3), #11 Training Script (D3), #12 Baseline Evaluation (D3)

## Data Loading

In [ ]:
# Setup Google Drive caching if on Colab
root = None
try:
    from src.colab_utils import setup_cabuaur_cached
    root = setup_cabuaur_cached()
    print("✓ Google Drive caching enabled")
except (ImportError, RuntimeError):
    print("Using default TorchGeo cache")

# Load dataloaders
print("\nLoading CaBuAr dataset...")
dataloaders = get_dataloaders(
    batch_size=4,  # Small batch for notebook (memory efficient)
    num_workers=0,
    normalize=True,
    root=root
)

print(f"Train samples: {len(dataloaders['datasets']['train'])}")
print(f"Val samples: {len(dataloaders['datasets']['val'])}")
print(f"Test samples: {len(dataloaders['datasets']['test'])}")

## Model and Loss Setup

In [ ]:
class BCEDiceLoss(nn.Module):
    """Hybrid loss combining BCE and Dice for class imbalance."""

    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, predictions, targets):
        # Extract burned class logits for BCE loss
        burned_logits = predictions[:, 1:2]  # [B, 1, H, W]
        bce_loss = self.bce(burned_logits, targets.float())

        # Dice loss
        probs = torch.softmax(predictions, dim=1)
        burned_prob = probs[:, 1:2]

        intersection = (burned_prob * targets).sum()
        union = (burned_prob + targets).sum()
        dice_loss = 1 - (2 * intersection + 1e-8) / (union + 1e-8)

        total_loss = self.bce_weight * bce_loss + self.dice_weight * dice_loss
        return total_loss


def compute_iou(predictions, targets, threshold=0.5):
    """Compute Intersection over Union for burned class."""
    with torch.no_grad():
        probs = torch.softmax(predictions, dim=1)
        burned_pred = (probs[:, 1] > threshold).float()
        targets_binary = targets.squeeze(1).float()

        intersection = (burned_pred * targets_binary).sum()
        union = (burned_pred + targets_binary).sum() - intersection
        iou = intersection / (union + 1e-8)

    return iou.item()


# Create model
print("Creating U-Net model...")
model = UNet(in_channels=24, out_channels=2).to(device)
print(f"Model parameters: {model.get_parameter_count():,}")

# Setup loss and optimizer
criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = Adam(model.parameters(), lr=0.0005)

## Training and Validation Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        batch = move_to_device(batch, device)
        images = batch['image']  # [B, 2, 12, 512, 512]
        masks = batch['mask']    # [B, 1, 512, 512]

        # Flatten timesteps: [B, 2, 12, 512, 512] -> [B, 24, 512, 512]
        B, T, C, H, W = images.shape
        images = images.view(B, T * C, H, W)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 5 == 0:
            print(f"  Batch {batch_idx + 1}/{len(train_loader)}: Loss = {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    return avg_loss


def validate(model, val_loader, criterion, device):
    """Validate model on validation set."""
    model.eval()
    total_loss = 0.0
    total_iou = 0.0

    with torch.no_grad():
        for batch in val_loader:
            batch = move_to_device(batch, device)
            images = batch['image']  # [B, 2, 12, 512, 512]
            masks = batch['mask']    # [B, 1, 512, 512]

            # Flatten timesteps
            B, T, C, H, W = images.shape
            images = images.view(B, T * C, H, W)

            outputs = model(images)
            loss = criterion(outputs, masks)
            iou = compute_iou(outputs, masks)

            total_loss += loss.item()
            total_iou += iou

    avg_loss = total_loss / len(val_loader)
    avg_iou = total_iou / len(val_loader)
    return avg_loss, avg_iou

## Training Loop

Configure and run training. Adjust `num_epochs` based on available time/compute.

**GPU (Colab):** 10-20 epochs takes ~5-10 minutes
**CPU:** Use 1-2 epochs for quick validation

In [ ]:
from ipywidgets import IntSlider, interact

# Interactive slider for epochs
epochs_slider = IntSlider(
    value=5,
    min=1,
    max=50,
    step=1,
    description='Epochs:',
    style={'description_width': '100px'}
)
display(epochs_slider)

In [ ]:
# Training configuration
num_epochs = epochs_slider.value
checkpoint_dir = Path('checkpoints_notebook')
checkpoint_dir.mkdir(exist_ok=True)

# History tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'val_iou': []
}
best_val_iou = 0.0

print("="*70)
print("U-Net Burn Detection Training")
print("="*70)
print(f"Epochs: {num_epochs}")
print(f"Device: {device}")
print(f"Checkpoint dir: {checkpoint_dir}")
print("="*70 + "\n")

# Training loop
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print("-" * 70)

    # Train
    train_loss = train_epoch(
        model, dataloaders['train'], criterion, optimizer, device
    )
    history['train_loss'].append(train_loss)
    print(f"Train Loss: {train_loss:.4f}")

    # Validate
    val_loss, val_iou = validate(
        model, dataloaders['val'], criterion, device
    )
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    print(f"Val Loss: {val_loss:.4f}, Val IoU: {val_iou:.4f}")

    # Checkpoint if best
    is_best = val_iou > best_val_iou
    if is_best:
        best_val_iou = val_iou
        best_path = checkpoint_dir / 'best_model.pth'
        torch.save(model.state_dict(), best_path)
        print(f"★ Saved best model: Val IoU = {best_val_iou:.4f}")

    print()

print("="*70)
print(f"Training complete!")
print(f"Best validation IoU: {best_val_iou:.4f}")
print("="*70)

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (BCE + Dice)')
axes[0].set_title('Training Progress: Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU curve
axes[1].plot(history['val_iou'], label='Val IoU (Burned Class)', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU Score')
axes[1].set_title('Validation Performance: IoU')
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTraining Summary:")
print(f"  Initial Val IoU: {history['val_iou'][0]:.4f}")
print(f"  Best Val IoU: {best_val_iou:.4f}")
print(f"  Final Val Loss: {history['val_loss'][-1]:.4f}")

## TODO: Implement Training

1. Define U-Net architecture (`src/unet.py`)
2. Load dataloaders from pipeline
3. Set up loss function (BCE + Dice)
4. Training loop with validation
5. Model checkpointing
6. Visualize training curves

In [ ]:
print("Training notebook - implementation coming soon")